# Notebook 01: Data Voorbereiden

In [9]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

BASE_DIR = Path.cwd().parent / "datasets iecon" / "datasets iecon" # Pad naar de datasets map
OUTPUT_DIR = Path.cwd().parent / "locatie_data"
OUTPUT_DIR.mkdir(exist_ok=True)

### 1. Data inladen en drie DataFrames maken (huis, middenhuis, carport)

In [4]:
def read_location(location_path):
    dfs = [] # Lege lijst waar de losse dfs in komen

    zip_files = list(location_path.rglob("*.zip")) # Zoekt in de datasets map en alle submappen naar alle bestanden die eindigen op .zip. rglob zorgt ervoor dat er ook in de submappen gezocht wordt. 

    print(f"{location_path.name}: {len(zip_files)} ZIP-bestanden gevonden") # Print hoeveel zip bestanden er gevonden zijn

    for i, zip_path in enumerate(zip_files, 1): # Loop door alle zip bestanden. enumerate(..,1) geeft elk bestand een teller mee die begint bij 1 (dit is voor de voortgang print per 500 bestanden hieronder.)
        with ZipFile(zip_path) as z: # opent 1 zip bestand tegelijk

            # vraagt alle bestandnamen op en houdt bestanden die eindigen op .csv 
            csv_files = [file for file in z.namelist()
                if file.endswith(".csv")]

            # loop door de csv bestanden binnen deze zip
            for csv_file in csv_files:

                # opent de csv direct zonder de zip eerst uit te pakken
                with z.open(csv_file) as f: 

                    df_csv = pd.read_csv(f)

                    # Hernoemt de eerste kolom (de datum & tijd) naar timestamp
                    df_csv = df_csv.rename(columns={"Unnamed: 0": "timestamp"})

                    # zet de timestamp kolom om in daadwerkelijke timestamp datatype met Amsterdam tijdzone. Pandas zet automatisch zomer/wintertijd om 
                    df_csv["timestamp"] = (
                        pd.to_datetime(
                            df_csv["timestamp"],
                            utc=True,
                            errors="coerce"
                        )
                        .dt.tz_convert("Europe/Amsterdam")
                    )

                    #voeg de dataframe toe aan de df lijst 
                    dfs.append(df_csv)

        # Elke 500 zip bestanden laat de code zien hoever hij is
        if i % 500 == 0:
            print(f"{i}/{len(zip_files)} verwerkt")

    # voegt alle losse dataframes uit dfs onder elkaar samen 
    return pd.concat(dfs, ignore_index=True)

In [5]:
df_huis_788 = read_location(BASE_DIR / "eiot-16eda211b788")
df_middenhuis_479 = read_location(BASE_DIR / "eiot-5335bae8d497")
df_carport = read_location(BASE_DIR / "eiot-e4393cd3dfdb")

eiot-16eda211b788: 7294 ZIP-bestanden gevonden
500/7294 verwerkt
1000/7294 verwerkt
1500/7294 verwerkt
2000/7294 verwerkt
2500/7294 verwerkt
3000/7294 verwerkt
3500/7294 verwerkt
4000/7294 verwerkt
4500/7294 verwerkt
5000/7294 verwerkt
5500/7294 verwerkt
6000/7294 verwerkt
6500/7294 verwerkt
7000/7294 verwerkt
eiot-5335bae8d497: 8556 ZIP-bestanden gevonden
500/8556 verwerkt
1000/8556 verwerkt
1500/8556 verwerkt
2000/8556 verwerkt
2500/8556 verwerkt
3000/8556 verwerkt
3500/8556 verwerkt
4000/8556 verwerkt
4500/8556 verwerkt
5000/8556 verwerkt
5500/8556 verwerkt
6000/8556 verwerkt
6500/8556 verwerkt
7000/8556 verwerkt
7500/8556 verwerkt
8000/8556 verwerkt
8500/8556 verwerkt
eiot-e4393cd3dfdb: 11447 ZIP-bestanden gevonden
500/11447 verwerkt
1000/11447 verwerkt
1500/11447 verwerkt
2000/11447 verwerkt
2500/11447 verwerkt
3000/11447 verwerkt
3500/11447 verwerkt
4000/11447 verwerkt
4500/11447 verwerkt
5000/11447 verwerkt
5500/11447 verwerkt
6000/11447 verwerkt
6500/11447 verwerkt
7000/11447 v

In [6]:
print(df_huis_788.shape)
print(df_middenhuis_479.shape)
print(df_carport.shape)

(3057890, 24)
(3493093, 24)
(4888431, 24)


In [10]:
df_huis_788.to_parquet(
    OUTPUT_DIR / "huis788.parquet", index=False
)

In [11]:
df_middenhuis_479.to_parquet(
    OUTPUT_DIR / "middenhuis479.parquet", index=False
)

In [12]:
df_carport.to_parquet(
    OUTPUT_DIR / "carport.parquet", index=False
)